In [1]:
# Load the dataset
# TotalCharges has some empty strings — convert them to numbers, then fill blanks with the median
import pandas as pd
df= pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [2]:
# These columns all have Yes/No answers — convert them to 1/0
yes_no_columns=['Partner','Dependents','PhoneService','OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport','StreamingTV',
'StreamingMovies','PaperlessBilling','Churn']
df[yes_no_columns]=df[yes_no_columns].apply(lambda x: x.str.strip().str.capitalize())
df[yes_no_columns]=df[yes_no_columns].replace({'Yes':1,'No':0, 'No internet service': 0})
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,1,0,1,0,No phone service,DSL,0,...,0,0,0,0,Month-to-month,1,Electronic check,29.85,29.85,0
1,5575-GNVDE,Male,0,0,0,34,1,No,DSL,1,...,1,0,0,0,One year,0,Mailed check,56.95,1889.50,0
2,3668-QPYBK,Male,0,0,0,2,1,No,DSL,1,...,0,0,0,0,Month-to-month,1,Mailed check,53.85,108.15,1
3,7795-CFOCW,Male,0,0,0,45,0,No phone service,DSL,1,...,1,1,0,0,One year,0,Bank transfer (automatic),42.30,1840.75,0
4,9237-HQITU,Female,0,0,0,2,1,No,Fiber optic,0,...,0,0,0,0,Month-to-month,1,Electronic check,70.70,151.65,1


In [3]:
# Encode gender as a number (Female=1, Male=2)
df.gender= df.gender.replace({'Female':1,'Male':2})

In [4]:
# These columns have multiple categories — use One-Hot Encoding to handle them
# drop='first' avoids the dummy variable trap
# Replace original columns with the encoded ones
from sklearn.preprocessing import OneHotEncoder
ohe=OneHotEncoder( drop='first')
columns_for_ohe=['MultipleLines','InternetService','Contract','PaymentMethod']
encoded_features = ohe.fit_transform(df[columns_for_ohe]).toarray()
encoded_df = pd.DataFrame(encoded_features, columns=ohe.get_feature_names_out(), index=df.index)
df = df.drop(columns=columns_for_ohe)
df = pd.concat([df, encoded_df], axis=1)
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,OnlineSecurity,OnlineBackup,DeviceProtection,...,Churn,MultipleLines_No phone service,MultipleLines_Yes,InternetService_Fiber optic,InternetService_No,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,7590-VHVEG,1,0,1,0,1,0,0,1,0,...,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,5575-GNVDE,2,0,0,0,34,1,1,0,1,...,0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
2,3668-QPYBK,2,0,0,0,2,1,1,1,0,...,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,7795-CFOCW,2,0,0,0,45,0,1,0,1,...,0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,9237-HQITU,1,0,0,0,2,1,0,0,0,...,1,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0


In [5]:
# Separate target (what we want to predict) from features (what we use to predict)
y=df.Churn.astype(int)
df=df.drop(['Churn','customerID'],axis='columns')

In [6]:
# Scale all features to a 0–1 range so no single column dominates
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
df = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)

In [7]:
# Split data — 80% for training, 20% for testing
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df, y, test_size=0.2)

In [8]:
# Try KNN with k from 1 to 20 and see which k gives the best test accuracy
from sklearn.neighbors import KNeighborsClassifier
for k in range(1,21):
    knn=KNeighborsClassifier(n_neighbors =k)
    knn.fit(X_train,y_train)
    print(k, knn.score(X_train, y_train), knn.score(X_test, y_test))

1 0.9984025559105432 0.7224982256919801
2 0.8665246716364927 0.7629524485450674
3 0.8640397586084487 0.7501774308019872
4 0.8349307774227902 0.7686302342086586
5 0.8313809016684416 0.7608232789212207
6 0.8212637557685482 0.7679205110007097
7 0.820376286829961 0.7665010645848119
8 0.8139865104721334 0.7714691270404542
9 0.8109691160809371 0.7665010645848119
10 0.8099041533546326 0.7750177430801988
11 0.8067092651757188 0.7799858055358411
12 0.80386936457224 0.7828246983676366
13 0.8036918707845225 0.7849538679914834
14 0.8061767838125665 0.7842441447835344
15 0.8031593894213702 0.7863733144073811
16 0.8022719204827831 0.7906316536550745
17 0.7990770323038694 0.7856635911994322
18 0.8008519701810437 0.7892122072391767
19 0.7987220447284346 0.7892122072391767
20 0.8024494142705005 0.794180269694819
